# Ders 02 - Microsoft Agent Framework'ü Keşfetmek

**Microsoft Agent Framework (MAF)**, yapay zeka ajanları oluşturmak için birleşik bir çerçevedir. Dört temel yapı taşı ile temiz, bileşenlere ayrılabilir bir mimari sağlar:

- **İstemci** – bir yapay zeka modeli uç noktasına bağlanır ve iletişimi yönetir
- **Ajan** – istemciyi talimatlar ve araç tanımları ile paketler
- **Araçlar** – modelin çağırabileceği özel işlevlerle ajanın yeteneklerini genişletir
- **Oturum** – çok turlu etkileşimler için konuşma geçmişini korur

Bu derste, bu kavramları kullanarak destinasyon uygunluğunu kontrol eden bir **seyahat rezervasyon ajanı** oluşturacağız.


## Kurulum


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Agent Çerçeve Mimarisi Anlama

Microsoft Agent Framework katmanlı bir mimariyi takip eder:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **İstemci** – Bir `AzureAIProjectAgentProvider`, bir Azure OpenAI dağıtımına bağlanır. Kimlik doğrulama, istek biçimlendirme ve yanıt ayrıştırmayı yönetir.
2. **Agent** – İstemciden `provider.create_agent()` aracılığıyla oluşturulur, agent model erişimini talimatlar (sistem istemi) ve araçlarla birleştirir.
3. **Araçlar** – Agent’ın eylem gerçekleştirmek veya veri almak için çağırabileceği `@tool` ile işaretlenmiş Python fonksiyonları.
4. **Oturum** – Konuşma geçmişini saklayan ve agent’ın önceki bağlamı hatırlayarak çok turlu diyaloğa olanak tanıyan `AgentSession` nesnesi (`agent.create_session()` ile oluşturulur).

Her katmanı adım adım oluşturalım.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool Dekoratörü ile Araç Ekleme

Araçlar, ajanların metin üretmenin ötesinde eylemler gerçekleştirmesine olanak tanır. `@tool` dekoratörü, normal bir Python fonksiyonunu ajanın çağırabileceği bir şeye dönüştürür.

Temel noktalar:
- Modelin her parametreyi anlaması için `Annotated[type, "description"]` kullanın.
- Docstring, modelin gördüğü araç açıklaması olur.
- `approval_mode="never_require"` aracı kullanıcı onayı olmadan otomatik olarak çalıştırır.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Araçlarla Bir Ajan Oluşturma

Şimdi istemciyi, talimatları ve araçları bir ajan içinde birleştiriyoruz. `instructions`, sistem istemi olarak görev yapar — ajanın kişiliğini ve davranışını tanımlar.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Oturumlarla Çok Turlu Konuşmalar

Bir `AgentSession` ( `agent.create_session()` ile oluşturulur) bir konuşmadaki tüm mesajları takip eder. Aynı oturumu her `agent.run()` çağrısına geçirerek, ajan tüm konuşma geçmişine erişebilir ve önceki mesajlara atıfta bulunabilir.

Her turda ajanın erişim durumu denetleyicimizi çağırabilmesi için `tools=[check_destination_availability]` geçiriyoruz.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Özet

Bu derste Microsoft Agent Framework'ün dört temel direğini keşfettiniz:

| Kavram | Öğrendikleriniz |
|---------|------------------|
| **İstemci** | `AzureAIProjectAgentProvider`, kimlik bilgisine dayalı kimlik doğrulama ile Azure OpenAI'ye bağlanır |
| **Agent** | `provider.create_agent()`, model bağlantısını talimatlar ve bir ad ile paketler |
| **Araçlar** | `@tool` dekoratörü, agentin çağırması için Python fonksiyonlarını açığa çıkarır |
| **Oturum** | `agent.create_session()`, birden çok tur boyunca konuşma geçmişini korur |

Bu yapı taşları bir araya gelerek doğal sohbetler yapabilen, harici fonksiyonları çağırabilen ve bağlamı sürdürebilen agentler oluşturur — sonraki derslerde daha gelişmiş agentik desenler için temel oluşturur.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstersek de, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayınız. Orijinal belge, kendi ana dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı nedeniyle oluşabilecek herhangi bir yanlış anlama veya yorum hatasından sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
